Market Entry Analysis for ABG Motors in India

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, roc_auc_score

In [2]:
# 1. Load the datasets
df_japan = pd.read_csv('JPN Data.xlsx - CN_Mobiles.csv')
df_india = pd.read_csv('IN_Data.xlsx - IN_Mobiles.csv')
print("Datasets loaded successfully!")



Datasets loaded successfully!


In [6]:
df_india


,ID,CURR_AGE,GENDER,ANN_INCOME,DT_MAINT
0,20710B05XL,54,M,"1,425,390",4/20/2018
1,89602T51HX,47,M,"1,678,954",6/8/2018
2,70190Z52IP,60,M,"931,624",7/31/2017
3,25623V15MU,55,F,"1,106,320",7/31/2017
4,36230I68CE,32,F,"748,465",1/27/2019
...,...,...,...,...,...
69995,35280V26PS,49,F,"861,770",4/8/2018
69996,79863N01VR,59,M,"888,976",5/17/2018
69997,94236O99QE,33,M,"699,676",4/21/2019
69998,23696V12DP,41,M,"1,881,922",7/14/2018


In [7]:
df_japan

,ID,CURR_AGE,GENDER,ANN_INCOME,AGE_CAR,PURCHASE
0,00001Q15YJ,50,M,"445,344",439,0
1,00003I71CQ,35,M,"107,634",283,0
2,00003N47FS,59,F,"502,787",390,1
3,00005H41DE,43,M,"585,664",475,0
4,00007E17UM,39,F,"705,723",497,1
...,...,...,...,...,...,...
39995,99988B18OG,49,M,"478,511",464,1
39996,99990Q44VP,28,M,"271,419",61,0
39997,99997Q09VJ,41,F,"590,704",379,1
39998,99998J59EE,50,M,"363,160",302,1


In [8]:
# 2. Initial Inspection (Using the Japanese dataset as our training baseline)
print("\n--- Japanese Dataset Info ---")
df_japan.info()


--- Japanese Dataset Info ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 40000 entries, 0 to 39999
Data columns (total 6 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   ID          40000 non-null  object
 1   CURR_AGE    40000 non-null  int64 
 2   GENDER      40000 non-null  object
 3   ANN_INCOME  40000 non-null  object
 4   AGE_CAR     40000 non-null  int64 
 5   PURCHASE    40000 non-null  int64 
dtypes: int64(3), object(3)
memory usage: 1.8+ MB


In [9]:
print("\n--- Summary Statistics ---")
display(df_japan.describe(include='all'))


--- Summary Statistics ---


,ID,CURR_AGE,GENDER,ANN_INCOME,AGE_CAR,PURCHASE
count,40000,40000.00000,40000,40000,40000.000000,40000.000000
unique,40000,NaN,2,38192,NaN,NaN
top,00001Q15YJ,NaN,M,"357,923",NaN,NaN
freq,1,NaN,22285,4,NaN,NaN
mean,NaN,44.99745,NaN,NaN,359.080250,0.575775
std,NaN,11.82008,NaN,NaN,203.063724,0.494231
min,NaN,25.00000,NaN,NaN,1.000000,0.000000
25%,NaN,35.00000,NaN,NaN,235.000000,0.000000
50%,NaN,45.00000,NaN,NaN,331.000000,1.000000
75%,NaN,55.00000,NaN,NaN,444.000000,1.000000


In [13]:
print("\n--- Japan Dataset Head ---")
print(df_japan.head())


--- Japan Dataset Head ---
           ID  CURR_AGE GENDER ANN_INCOME  AGE_CAR  PURCHASE
0  00001Q15YJ        50      M    445,344      439         0
1  00003I71CQ        35      M    107,634      283         0
2  00003N47FS        59      F    502,787      390         1
3  00005H41DE        43      M    585,664      475         0
4  00007E17UM        39      F    705,723      497         1


In [10]:
# 3. Check for Missing Values
print("\n--- Missing Values in Japanese Dataset ---")
print(df_japan.isnull().sum())


--- Missing Values in Japanese Dataset ---
ID            0
CURR_AGE      0
GENDER        0
ANN_INCOME    0
AGE_CAR       0
PURCHASE      0
dtype: int64


In [14]:
print("\n--- India Dataset Info ---")
df_india.info()


--- India Dataset Info ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 70000 entries, 0 to 69999
Data columns (total 5 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   ID          70000 non-null  object
 1   CURR_AGE    70000 non-null  int64 
 2   GENDER      70000 non-null  object
 3   ANN_INCOME  70000 non-null  object
 4   DT_MAINT    70000 non-null  object
dtypes: int64(1), object(4)
memory usage: 2.7+ MB


In [15]:
print("\n--- India Dataset Head ---")
print(df_india.head())
print("\n--- India Missing Values ---")
print(df_india.isnull().sum())


--- India Dataset Head ---
           ID  CURR_AGE GENDER    ANN_INCOME   DT_MAINT
0  20710B05XL        54      M    1,425,390   4/20/2018
1  89602T51HX        47      M    1,678,954    6/8/2018
2  70190Z52IP        60      M      931,624   7/31/2017
3  25623V15MU        55      F    1,106,320   7/31/2017
4  36230I68CE        32      F      748,465   1/27/2019

--- India Missing Values ---
ID            0
CURR_AGE      0
GENDER        0
ANN_INCOME    0
DT_MAINT      0
dtype: int64


In [12]:
# 4. Outlier Detection (Example using Interquartile Range - IQR for a numerical feature)
def handle_outliers(df, column):
    Q1 = df[column].quantile(0.25)
    Q3 = df[column].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    
    # Capping outliers
    df[column] = np.where(df[column] > upper_bound, upper_bound, df[column])
    df[column] = np.where(df[column] < lower_bound, lower_bound, df[column])
    return df

In [18]:
print("Japanese AGE_CAR stats:")
print(df_japan['AGE_CAR'].describe())

df_india['DT_MAINT'] = pd.to_datetime(df_india['DT_MAINT'])
print("\nIndian DT_MAINT stats:")
print(df_india['DT_MAINT'].describe())

Japanese AGE_CAR stats:
count    40000.000000
mean       359.080250
std        203.063724
min          1.000000
25%        235.000000
50%        331.000000
75%        444.000000
max       1020.000000
Name: AGE_CAR, dtype: float64

Indian DT_MAINT stats:
count                            70000
mean     2018-06-28 16:10:28.662856960
min                2016-09-14 00:00:00
25%                2018-03-15 00:00:00
50%                2018-07-26 00:00:00
75%                2018-12-24 00:00:00
max                2019-06-30 00:00:00
Name: DT_MAINT, dtype: object


In [19]:
# Assuming a reference date for the Indian dataset to calculate AGE_CAR
ref_date = pd.to_datetime('2020-01-01') # Just testing a date to see if the days align
df_india['AGE_CAR_est'] = (ref_date - df_india['DT_MAINT']).dt.days
print("\nIndian estimated AGE_CAR stats with reference 2020-01-01:")
print(df_india['AGE_CAR_est'].describe())


Indian estimated AGE_CAR stats with reference 2020-01-01:
count    70000.000000
mean       551.326057
std        241.999792
min        185.000000
25%        373.000000
50%        524.000000
75%        657.000000
max       1204.000000
Name: AGE_CAR_est, dtype: float64


In [20]:
df_india['DT_MAINT'] = pd.to_datetime(df_india['DT_MAINT'])
print("\nIndian DT_MAINT min:", df_india['DT_MAINT'].min())
print("Indian DT_MAINT max:", df_india['DT_MAINT'].max())


Indian DT_MAINT min: 2016-09-14 00:00:00
Indian DT_MAINT max: 2019-06-30 00:00:00


In [21]:
# Let's calculate difference from a reference date, e.g. end of 2019
ref_date = pd.to_datetime('2019-12-31')
print(f"\nIndian estimated AGE_CAR stats with reference {ref_date.date()}:")
print((ref_date - df_india['DT_MAINT']).dt.days.describe())


Indian estimated AGE_CAR stats with reference 2019-12-31:
count    70000.000000
mean       550.326057
std        241.999792
min        184.000000
25%        372.000000
50%        523.000000
75%        656.000000
max       1203.000000
Name: DT_MAINT, dtype: float64


In [22]:
df_india = pd.read_csv("IN_Data.xlsx - IN_Mobiles.csv")
df_india['DT_MAINT'] = pd.to_datetime(df_india['DT_MAINT'])

In [23]:
ref_date2 = df_india['DT_MAINT'].max()
print(f"Reference date {ref_date2}:")
print((ref_date2 - df_india['DT_MAINT']).dt.days.describe())

Reference date 2019-06-30 00:00:00:
count    70000.000000
mean       366.326057
std        241.999792
min          0.000000
25%        188.000000
50%        339.000000
75%        472.000000
max       1019.000000
Name: DT_MAINT, dtype: float64


In [24]:
# Clean Japan
df_japan['ANN_INCOME'] = df_japan['ANN_INCOME'].str.replace(',', '').astype(float)
df_japan['GENDER'] = df_japan['GENDER'].map({'M': 1, 'F': 0})

In [27]:
#  Clean India
df_india['ANN_INCOME'] = df_india['ANN_INCOME'].astype(str).str.replace(',', '').astype(float)
df_india['GENDER'] = df_india['GENDER'].map({'M': 1, 'F': 0})
df_india['DT_MAINT'] = pd.to_datetime(df_india['DT_MAINT'])
df_india['AGE_CAR'] = (df_india['DT_MAINT'].max() - df_india['DT_MAINT']).dt.days

In [30]:
# . Train Model
features = ['CURR_AGE', 'GENDER', 'ANN_INCOME', 'AGE_CAR']
X = df_japan[features]
y = df_japan['PURCHASE']

X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)

model = LogisticRegression()
model.fit(X_train_scaled, y_train)

,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,True
,intercept_scaling,1
,class_weight,None
,random_state,None
,solver,'lbfgs'
,max_iter,100
,multi_class,'deprecated'


In [31]:
# Metrics
preds = model.predict(X_val_scaled)
print(classification_report(y_val, preds))
print("ROC-AUC:", roc_auc_score(y_val, model.predict_proba(X_val_scaled)[:, 1]))

              precision    recall  f1-score   support

           0       0.64      0.55      0.59      3349
           1       0.71      0.78      0.74      4651

    accuracy                           0.68      8000
   macro avg       0.68      0.66      0.67      8000
weighted avg       0.68      0.68      0.68      8000

ROC-AUC: 0.7369598963135999


In [41]:
# Coefficients
for col, coef in zip(features, model.coef_[0]):
    print(f"{col}: {coef:.4f}")

CURR_AGE: -0.1262
GENDER: 0.1089
ANN_INCOME: 0.4039
AGE_CAR: 0.8464


In [46]:
# 5. Predict India
df_india_clean = df_india.copy()

# Convert features to numbers and replace invalid data/NaNs with 0
for col in features:
    df_india_clean[col] = pd.to_numeric(df_india_clean[col], errors='coerce')
    df_india_clean[col] = df_india_clean[col].fillna(0)  # Keeps rows from getting deleted

# Extract features safely
X_india = df_india_clean[features]

# Scale and predict
X_india_scaled = scaler.transform(X_india)
X_india_scaled_df = pd.DataFrame(X_india_scaled, columns=features)
india_preds = model.predict(X_india_scaled_df)

print("India potential customers:", sum(india_preds))

India potential customers: 66709


c:\Users\AMIT KUMAR\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\utils\validation.py:2742: UserWarning: X has feature names, but LogisticRegression was fitted without feature names
  warnings.warn(


In [47]:
print(df_japan['PURCHASE'].value_counts(normalize=True))
print(df_india['ANN_INCOME'].describe())
print(df_japan['ANN_INCOME'].describe())
print(df_india['AGE_CAR'].describe())
print(df_japan['AGE_CAR'].describe())

PURCHASE
1    0.575775
0    0.424225
Name: proportion, dtype: float64
count    7.000000e+04
mean     1.148679e+06
std      3.994505e+05
min      3.000330e+05
25%      8.568238e+05
50%      1.125152e+06
75%      1.438676e+06
max      1.999989e+06
Name: ANN_INCOME, dtype: float64
count     40000.000000
mean     359398.878225
std      175109.260472
min       70089.000000
25%      219766.000000
50%      337657.000000
75%      464260.750000
max      799971.000000
Name: ANN_INCOME, dtype: float64
count    70000.000000
mean       366.326057
std        241.999792
min          0.000000
25%        188.000000
50%        339.000000
75%        472.000000
max       1019.000000
Name: AGE_CAR, dtype: float64
count    40000.000000
mean       359.080250
std        203.063724
min          1.000000
25%        235.000000
50%        331.000000
75%        444.000000
max       1020.000000
Name: AGE_CAR, dtype: float64


In [49]:
# . Data Cleaning: Japanese Dataset
# Remove commas from income and convert to float
df_japan['ANN_INCOME'] = df_japan['ANN_INCOME'].astype(str).str.replace(',', '').astype(float)
# Convert Gender to numeric (M = 1, F = 0)
df_japan['GENDER'] = df_japan['GENDER'].map({'M': 1, 'F': 0})

In [51]:
#  Data Cleaning & Feature Engineering: Indian Dataset
df_india['ANN_INCOME'] = df_india['ANN_INCOME'].astype(str).str.replace(',', '').astype(float)
df_india['GENDER'] = df_india['GENDER'].map({'M': 1, 'F': 0})

In [52]:
# Engineer 'AGE_CAR' from 'DT_MAINT'
df_india['DT_MAINT'] = pd.to_datetime(df_india['DT_MAINT'])
# We use the most recent date in the dataset as the reference point to calculate age in days
reference_date = df_india['DT_MAINT'].max() 
df_india['AGE_CAR'] = (reference_date - df_india['DT_MAINT']).dt.days

In [53]:
# Check the results
print("Japanese Dataset Summary:")
print(df_japan[['CURR_AGE', 'ANN_INCOME', 'AGE_CAR']].describe())
print("\nIndian Dataset Summary:")
print(df_india[['CURR_AGE', 'ANN_INCOME', 'AGE_CAR']].describe())

Japanese Dataset Summary:
          CURR_AGE     ANN_INCOME       AGE_CAR
count  40000.00000   40000.000000  40000.000000
mean      44.99745  359398.878225    359.080250
std       11.82008  175109.260472    203.063724
min       25.00000   70089.000000      1.000000
25%       35.00000  219766.000000    235.000000
50%       45.00000  337657.000000    331.000000
75%       55.00000  464260.750000    444.000000
max       65.00000  799971.000000   1020.000000

Indian Dataset Summary:
           CURR_AGE    ANN_INCOME       AGE_CAR
count  70000.000000  7.000000e+04  70000.000000
mean      44.995314  1.148679e+06    366.326057
std       11.822122  3.994505e+05    241.999792
min       25.000000  3.000330e+05      0.000000
25%       35.000000  8.568238e+05    188.000000
50%       45.000000  1.125152e+06    339.000000
75%       55.000000  1.438676e+06    472.000000
max       65.000000  1.999989e+06   1019.000000


In [ ]:
# Define Features (X) and Target (y)
features = ['CURR_AGE', 'GENDER', 'ANN_INCOME', 'AGE_CAR']

# 1. Clean the target variable first (drop only if PURCHASE is missing)
df_japan_clean = df_japan.dropna(subset=['PURCHASE']).copy()

# 2. Fill missing numeric values with the median instead of deleting the rows
for col in ['CURR_AGE', 'ANN_INCOME', 'AGE_CAR']:
    median_value = df_japan_clean[col].median()
    df_japan_clean[col] = df_japan_clean[col].fillna(median_value)

# 3. Handle GENDER column missing values if any (fill with most frequent value)
if df_japan_clean['GENDER'].dtype == 'object':
    most_frequent = df_japan_clean['GENDER'].mode()[0]
    df_japan_clean['GENDER'] = df_japan_clean['GENDER'].fillna(most_frequent)

# 4. Convert categorical strings to numbers safely
X = pd.get_dummies(df_japan_clean[features], drop_first=True)
y = df_japan_clean['PURCHASE']

# Now your train_test_split will have plenty of rows to work with!
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

In [67]:


# Define Features (X) and Target (y)
features = ['CURR_AGE', 'GENDER', 'ANN_INCOME', 'AGE_CAR']

# --- STEP 1: DROP ROWS MISSING THE TARGET ---
df_japan_clean = df_japan.dropna(subset=['PURCHASE']).copy()

# --- STEP 2: HANDLE CATEGORICAL ENCODING FIRST ---
# If GENDER is text (e.g., Male/Female), this converts it to numeric 0 and 1 columns
X_encoded = pd.get_dummies(df_japan_clean[features], columns=['GENDER'], drop_first=True)

# --- STEP 3: FORCE NUMERIC CONVERSION ON THE REST & FILL NaNs ---
# Convert everything in our features matrix to numeric float values
X_encoded = X_encoded.apply(pd.to_numeric, errors='coerce')

# Fill any remaining NaNs with the column median
for col in X_encoded.columns:
    median_val = X_encoded[col].median()
    if pd.isna(median_val):
        median_val = 0
    X_encoded[col] = X_encoded[col].fillna(median_val)

y = df_japan_clean['PURCHASE'].astype(int)

# --- STEP 4: TRAIN/VALIDATION SPLIT ---
X_train, X_val, y_train, y_val = train_test_split(X_encoded, y, test_size=0.2, random_state=42)

# --- STEP 5: FEATURE SCALING ---
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)

# --- STEP 6: MODEL TRAINING ---
model = LogisticRegression(random_state=42)
model.fit(X_train_scaled, y_train)

print("Model trained successfully!")

Model trained successfully!


In [68]:
accuracy = model.score(X_val_scaled, y_val)
print(f"Validation Accuracy: {accuracy * 100:.2f}%")

Validation Accuracy: 68.58%


In [70]:
# 1. Predictions on Validation Set
y_pred = model.predict(X_val_scaled)
y_prob = model.predict_proba(X_val_scaled)[:, 1]

# 2. Evaluation Metrics
print("Classification Report (Accuracy, Precision, Recall, F1-Score):")
print(classification_report(y_val, y_pred))

print(f"ROC-AUC Score: {roc_auc_score(y_val, y_prob):.4f}\n")

Classification Report (Accuracy, Precision, Recall, F1-Score):
              precision    recall  f1-score   support

           0       0.65      0.55      0.60      3349
           1       0.71      0.78      0.74      4651

    accuracy                           0.69      8000
   macro avg       0.68      0.67      0.67      8000
weighted avg       0.68      0.69      0.68      8000

ROC-AUC Score: 0.7359



In [71]:
# 3. Coefficient Interpretation
print("Business Influence (Coefficients):")
for col, coef in zip(features, model.coef_[0]):
    print(f"{col}: {coef:.4f}")

Business Influence (Coefficients):
CURR_AGE: -0.1263
GENDER: 0.3975
ANN_INCOME: 0.8415


In [73]:
# 1. Prepare Indian features aligned EXACTLY with the scaler's training data
# Using scaler.feature_names_in_ guarantees the correct order and columns
X_india = df_india[scaler.feature_names_in_]

# 2. Scale the Indian data using the SAME scaler fitted on the Japanese data
X_india_scaled = scaler.transform(X_india)

# 3. Predict purchases
india_predictions = model.predict(X_india_scaled)
df_india['PREDICTED_PURCHASE'] = india_predictions

# 4. Calculate total potential customers
total_potential_sales = india_predictions.sum()
print(f"Total projected sales in the Indian sample: {total_potential_sales}")

Total projected sales in the Indian sample: 67356
